# Load Data

In [1]:
import pandas as pd

data=pd.read_csv('data/processed/creditcard_clean.csv')
df=data.copy()
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,-0.995290,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,1.774718,0
1,-0.995290,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,-0.268530,0
2,-0.995279,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,4.959811,0
3,-0.995279,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,1.411487,0
4,-0.995267,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,0.667362,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
283721,1.035258,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,-0.295230,0
283722,1.035270,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,0.038798,0
283723,1.035282,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,0.638020,0
283724,1.035282,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,-0.166875,0


In [2]:
from sklearn.model_selection import train_test_split

X = df.drop('Class', axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts()}")
print(f"Test class distribution:\n{y_test.value_counts()}")

Train shape: (226980, 30), Test shape: (56746, 30)
Train class distribution:
Class
0    226602
1       378
Name: count, dtype: int64
Test class distribution:
Class
0    56651
1       95
Name: count, dtype: int64


# Log experiments, parameters, and metrics.

In [15]:
import mlflow
mlflow.set_experiment("Project")

2026/04/21 15:56:23 INFO mlflow.tracking.fluent: Experiment with name 'Project' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:c:/Users/PC/OneDrive - ESPRIT/Bureau/MLOps-Project/mlruns/3', creation_time=1776801383615, experiment_id='3', last_update_time=1776801383615, lifecycle_stage='active', name='Project', tags={}, trace_location=None, workspace='default'>

In [16]:
import warnings
warnings.filterwarnings('ignore')

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

models = [

    (
        "Logistic Regression", 
        {
            "solver": "liblinear",
            "max_iter": 1000,
            "random_state": 42
        },
        LogisticRegression(),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "Random Forest", 
        {
            "n_estimators": 100,
            "random_state": 13
        },
        RandomForestClassifier(),
        (X_train, y_train),
        (X_test, y_test)
    ),

    (
        "KNN (k=3)", 
        {
            "n_neighbors": 3
        },
        KNeighborsClassifier(),
        (X_train, y_train),
        (X_test, y_test)
    ),

]

In [31]:
from sklearn.metrics import classification_report
import mlflow
import mlflow.sklearn

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]

    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(params)
        mlflow.log_metrics({
            "accuracy": report["accuracy"] * 100,
            "precision": report["1"]["precision"] * 100,
            "recall": report["1"]["recall"] * 100,
            "recall_class_1": report["1"]["recall"] * 100,
            "recall_class_0": report["0"]["recall"] * 100,
            "f1_score_macro": report["macro avg"]["f1-score"] * 100
        })

        mlflow.sklearn.log_model(model, model_name)

2026/04/22 13:42:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 13:42:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/22 13:46:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/22 13:46:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

# Compare different models and runs.

In [20]:
import mlflow

# Get experiment
experiment = mlflow.get_experiment_by_name("Project")

# Fetch all runs as pandas DataFrame
runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

# Keep useful columns
runs_df = runs_df[[
    "tags.mlflow.runName",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.recall_class_1",
    "metrics.recall_class_0",
    "metrics.f1_score_macro"
]]

print("All runs:")
display(runs_df)

# Compare models (descending) for each metric
metrics = [
    "accuracy",
    "precision",
    "recall",
    "recall_class_1",
    "recall_class_0",
    "f1_score_macro"
]

for m in metrics:
    print("\n==============================")
    print("Ranking by:", m)
    
    display(
        runs_df.sort_values(
            f"metrics.{m}",
            ascending=False
        )[[
            "tags.mlflow.runName",
            f"metrics.{m}"
        ]]
    )

All runs:


,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.recall_class_1,metrics.recall_class_0,metrics.f1_score_macro
0,KNN (k=3),99.947133,95.774648,71.578947,71.578947,99.994704,90.950619
1,Random Forest,99.950657,97.183099,72.631579,72.631579,99.996470,91.553911
2,Logistic Regression,99.911888,84.615385,57.894737,57.894737,99.982348,84.352941



Ranking by: accuracy


,tags.mlflow.runName,metrics.accuracy
1,Random Forest,99.950657
0,KNN (k=3),99.947133
2,Logistic Regression,99.911888



Ranking by: precision


...

,tags.mlflow.runName,metrics.recall_class_1
1,Random Forest,72.631579
0,KNN (k=3),71.578947
2,Logistic Regression,57.894737



Ranking by: recall_class_0


,tags.mlflow.runName,metrics.recall_class_0
1,Random Forest,99.996470
0,KNN (k=3),99.994704
2,Logistic Regression,99.982348



Ranking by: f1_score_macro


,tags.mlflow.runName,metrics.f1_score_macro
1,Random Forest,91.553911
0,KNN (k=3),90.950619
2,Logistic Regression,84.352941


#  Identify and select a champion model

In [32]:
top_models = mlflow.search_logged_models(
    experiment_ids=[experiment.experiment_id],
    order_by=[{"field_name": "metrics.accuracy", "ascending": False}],
    max_results=1,
)

best_model = top_models.iloc[0]

print("Champion model:")
print(best_model)

Champion model:
artifact_location         file:c:/Users/PC/OneDrive - ESPRIT/Bureau/MLOp...
creation_timestamp                                            1776879968661
experiment_id                                                             3
last_updated_timestamp                                        1776879974493
metrics                   [<Metric: dataset_digest=None, dataset_name=No...
model_id                                 m-5873e5c464574786bf320c4ba6049347
model_type                                                             None
name                                                          Random Forest
params                        {'n_estimators': '100', 'random_state': '13'}
source_run_id                              c51d7f04e9f64ed2bb1f9d6b82422d05
status                                                                READY
status_message                                                         None
tags                      {'mlflow.source.name': 'mlflow_experiment_trac

In [33]:
import os
os.environ["MLFLOW_LOCK_MODEL_DEPENDENCIES"] = "true"

In [34]:
registered = mlflow.register_model(
    model_uri=f"models:/{best_model.model_id}",
    name="best_fraud_model",
)

print("registered version:", registered.version)

registered version: 2


Registered model 'best_fraud_model' already exists. Creating a new version of this model...
Created version '2' of model 'best_fraud_model'.


In [35]:
from mlflow.tracking import MlflowClient

# assign alias champion
client = MlflowClient()

client.set_registered_model_alias(
    name="best_fraud_model",
    alias="champion",
    version=registered.version,
)

In [36]:
champion_model = mlflow.sklearn.load_model(
    model_uri="models:/best_fraud_model@champion"
)
print(champion_model)

RandomForestClassifier(random_state=13)
